Mount google drive folder

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


SetUp Working Directory

In [ ]:
import sys
# Path to the directory *containing* the utils folder
sys.path.insert(0, '...')

Import libraries

In [ ]:
!pip install gensim

In [ ]:
import nltk
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('wordnet')

In [ ]:
import os
from pathlib import Path
import joblib as jl
import numpy as np
import polars as pl
import pandas as pd
from sklearn.model_selection import train_test_split
from collections import defaultdict
from gensim.models import Word2Vec
import time
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score

Import Custom Libraries

In [ ]:
!pip install contractions
!pip install unidecode

In [ ]:
from utils import pipeline_v2, funcs
from utils.funcs import cooccurrence_matrix_generator, song_vector, diagnose
from utils.pipeline_v2 import main_pipeline

SetUp Data Directory

In [ ]:
data_dir = Path('...')

Read Dataset

In [ ]:
df_train = pl.read_csv(data_dir / 'train.csv')
df_test = pl.read_csv(data_dir / 'test.csv')

### Removing Duplicate Songs

Lyrics sites often contain the same song more than once (covers, remasters, re-uploads). If one copy ends up in the training split and another in the validation split, the model is rewarded for memorizing instead of generalizing. Exact duplicates are therefore removed from the training data **before** any cleaning or splitting. The number of test lyrics that also appear in the training data is printed for reference only; test rows are never dropped.

In [ ]:
print("train rows before:", df_train.height)
print("duplicate lyrics in train:", df_train["lyrics"].is_duplicated().sum())
print("test lyrics also in train:", df_test["lyrics"].is_in(df_train["lyrics"].implode()).sum())

# keep the first copy of every song; the test set is left untouched because every id needs a prediction
df_train = df_train.unique(subset="lyrics", keep="first", maintain_order=True)
print("train rows after:", df_train.height)

train rows before: 134967
duplicate lyrics in train: 343
test lyrics also in train: 77
train rows after: 134789


In [ ]:
sample_df1, sample_df2 = train_test_split(
    df_train,
    train_size=0.5,
    stratify=df_train['tag'],
    random_state=80085
)

### Lyrics Preprocessing Overview

Each partition is mapped through `main_pipeline` via Polars' `map_elements` to clean and standardize the `lyrics` column:

- **Text Cleaning & Normalization:** Strips song metadata tags (e.g., *[Chorus]*, *[Verse]*), URLs, brackets, and punctuation, while expanding contractions.
- **Filtering & Tokenization:** Tokenizes text into individual words, discards non-alphabetic tokens, and removes standard English stopwords as well as common musical filler vocalizations (e.g., *"oh"*, *"yeah"*, *"la"*).
- **Linguistic Standardization:** Converts diacritics to ASCII, applies lowercasing, and lemmatizes tokens across multiple parts of speech.
- **Output Schema:** With `tokenized_output=True`, each song's lyrics are transformed into a list of cleaned tokens (`pl.List(pl.String)`) stored under `cleaned_lyrics`.

Once both batches finish processing, they are combined back into a single unified dataset (`cleaned_df`) using vertical concatenation (`pl.concat`).

In [ ]:
sample_df1 = sample_df1.with_columns(
    pl.col("lyrics")
    .map_elements(
        lambda x: main_pipeline(x, tokenized_output=True), #set to False for transformer
        return_dtype=pl.List(pl.String)#if tokenized_output=False this line should be removed
    )
    .alias("cleaned_lyrics")
)

sample_df2 = sample_df2.with_columns(
    pl.col("lyrics")
    .map_elements(
        lambda x: main_pipeline(x, tokenized_output=True), #set to False for transformer
        return_dtype=pl.List(pl.String) #if tokenized_output=False this line should be removed
    )
    .alias("cleaned_lyrics")
)

cleaned_df_train = pl.concat([sample_df1, sample_df2])

cleaned_df_test = df_test.with_columns(
    pl.col("lyrics")
    .map_elements(
        lambda x: main_pipeline(x, tokenized_output=True), #set to False for transformer
        return_dtype=pl.List(pl.String)#if tokenized_output=False this line should be removed
    )
    .alias("cleaned_lyrics"))

Remove little to no lyrics entries (<10)

In [ ]:
print("Before filtering - train rows:", cleaned_df_train.height)

# Filter out songs with fewer than 10 tokens
cleaned_df_train = cleaned_df_train.filter(pl.col("cleaned_lyrics").list.len() >= 10)

print("After filtering - train rows:", cleaned_df_train.height)

Before filtering - train rows: 134789
After filtering - train rows: 134582


## Checkpointing & Class Distribution Overview

To preserve our preprocessed data and avoid redundant recomputation, we export the DataFrame to **Parquet format** and reload it. Parquet provides efficient column-oriented compression and natively preserves complex nested schemas, such as columns containing lists of strings (`pl.List(pl.String)`).

An initial inspection of the target genre distribution reveals a notable class imbalance, with **Rock**, **Pop**, and **Rap** making up the substantial majority of the dataset. Downstream modeling and evaluation strategies will need to account for this skew.

In [ ]:
#IGNORE THIS
# transformers get plain cleaned text (tokens joined by spaces) and keep the id
#cleaned_df_train_tran.select(['id', 'tag', pl.col('cleaned_lyrics').list.join(' ')]).write_parquet(tran_dir / 'cleaned_train_df.parquet')
#cleaned_df_test_tran.select(['id', pl.col('cleaned_lyrics').list.join(' ')]).write_parquet(tran_dir / 'cleaned_test_df.parquet')

In [ ]:
w2v_dir = Path('...')

In [ ]:
cleaned_df_train.select(['id', 'tag', 'cleaned_lyrics']).write_parquet(w2v_dir / 'cleaned_train_df_tokenized.parquet')
cleaned_df_test.select(['id', 'cleaned_lyrics']).write_parquet(w2v_dir / 'cleaned_test_df_tokenized.parquet')

In [ ]:
cleaned_df_train = pl.read_parquet(w2v_dir / 'cleaned_train_df_tokenized.parquet')
cleaned_df_test = pl.read_parquet(w2v_dir / 'cleaned_test_df_tokenized.parquet')

In [ ]:
cleaned_df_train['tag'].value_counts()

tag,count
str,u32
"""misc""",5601
"""rock""",25282
"""country""",3461
"""rb""",6191
"""pop""",55508
"""rap""",38539


## Train/Validation Split

The cleaned training songs are split 75/25, stratified by genre, so the validation set keeps the natural genre distribution. Everything that learns from data (co-occurrence matrix, GloVe, TF-IDF) is fitted on `X_train` only. The song ids of both parts are saved to `data/splits.parquet`, so every model (random forest, coevolution forest, transformer) can use exactly the same validation set.

In [ ]:
X = cleaned_df_train.select('id', 'cleaned_lyrics')
y = cleaned_df_train.select('tag')

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.25, random_state=80085, stratify=y
)

In [ ]:
# which song went where, in the same row order as the feature files
pl.concat([
    X_train.select("id").with_columns(pl.lit("train").alias("split")),
    X_val.select("id").with_columns(pl.lit("val").alias("split")),
]).write_parquet(data_dir / "splits.parquet")

---
# ▼ Word2Vec parameter selection starts here ▼
---

**Everything above this cell follows the same steps as `preprocessing_unified.ipynb` and can be merged with it** (data loading, duplicate removal, cleaning, short-song filter, checkpoint, train/validation split). Two details differ: this notebook uses `pipeline_v2`, the unified one `pipeline_v3`, and this notebook keeps songs with at least 10 tokens (`>= 10`), the unified one more than 10 (`> 10`). After merging, the cleaned lyrics and the split change slightly, so rerun the tuning if the numbers must match exactly.

When merging, keep the Word2Vec-specific imports in the setup cells (`gensim`, `time`, `LogisticRegression`, `accuracy_score`, `f1_score`, `song_vector`).

**Only the cells below are specific to Word2Vec tuning:** labels → logistic regression probe → 14 configurations → run all → chosen configuration.

## Word2Vec song embeddings (lyrics only)
Skip-gram trained on the training lyrics only (no validation or test songs, no metadata).
Each song is represented by the mean of its word vectors, which is the feature matrix for the classifier.

### Labels

The genre labels are extracted from the one-column Polars frames as flat 1D NumPy arrays (one genre per song, same row order as `X_train` / `X_val`), the shape scikit-learn expects. The labels stay as genre names.

In [ ]:
y_train = y_train["tag"].to_numpy()
y_val   = y_val["tag"].to_numpy()

### Tuning with a logistic regression

Each configuration trains Word2Vec on the full `X_train` lyrics, builds mean song vectors, fits logistic regression with balanced class weights and scores it on the full validation set. The probe is only a tuning tool.

In [ ]:
def run_config(name, cfg):
    """Train Word2Vec with `cfg`, fit the probe on mean song vectors and score it on the validation set."""
    start = time.time()
    w2v = Word2Vec(sentences=X_train["cleaned_lyrics"].to_list(), seed=80085, **cfg)
    vectors, index = w2v.wv.vectors, w2v.wv.key_to_index

    def embed(df):
        return np.vstack([song_vector(t, vectors, index) for t in df["cleaned_lyrics"].to_list()])

    X_tr, X_va = embed(X_train), embed(X_val)
    # Validation songs without a single in-vocabulary word get a zero vector
    zero_va = sum(all(t not in index for t in s) for s in X_val["cleaned_lyrics"].to_list())

    probe = LogisticRegression(max_iter=1000, class_weight="balanced")
    probe.fit(X_tr, y_train)
    pred = probe.predict(X_va)

    return {"changed": name, **cfg, "vocab": len(w2v.wv), "zero_val_songs": zero_va,
            "accuracy": accuracy_score(y_val, pred),
            "macro_f1": f1_score(y_val, pred, average="macro"),
            "weighted_f1": f1_score(y_val, pred, average="weighted"),
            "seconds": round(time.time() - start)}

### Configurations

The baseline uses the gensim defaults, except for Skip-gram (`sg=1`). The configurations are:

1. the baseline,
2. CBOW (`sg=0`) with everything else at baseline,
3. each parameter lowered on its own,
4. each parameter raised on its own.

`workers` stays at 3 and is not varied, since it affects speed, not quality.

In [ ]:
default = dict(sg=1, vector_size=100, window=5, min_count=5, negative=5, epochs=5, sample=1e-3, workers=3)

configs = [
    ("default",         default),
    ("sg=0 (CBOW)",     {**default, "sg": 0}),

    # lowered
    ("vector_size=50",  {**default, "vector_size": 50}),
    ("window=3",        {**default, "window": 3}),
    ("min_count=2",     {**default, "min_count": 2}),
    ("negative=2",      {**default, "negative": 2}),
    ("epochs=2",        {**default, "epochs": 2}),
    ("sample=0.0001",   {**default, "sample": 1e-4}),

    # raised
    ("vector_size=300", {**default, "vector_size": 300}),
    ("window=10",       {**default, "window": 10}),
    ("min_count=10",    {**default, "min_count": 10}),
    ("negative=10",     {**default, "negative": 10}),
    ("epochs=10",       {**default, "epochs": 10}),
    ("sample=0.01",     {**default, "sample": 1e-2}),
]
print(len(configs), "configurations")

### Run all configurations

After every iteration, results are written to `word2vec_param_testing.csv` in `w2v_dir`.

In [ ]:
csv_path = w2v_dir / "word2vec_param_testing.csv"
rows = []

for i, (name, cfg) in enumerate(configs, 1):
    print(f"[{i}/{len(configs)}] {name}")
    rows.append(run_config(name, cfg))
    pl.DataFrame(rows).write_csv(csv_path)
    print(f"    macro F1 = {rows[-1]['macro_f1']:.3f} | accuracy = {rows[-1]['accuracy']:.3f} | {rows[-1]['seconds']}s")

tuning_results = pl.DataFrame(rows).sort("macro_f1", descending=True)
tuning_results

### Chosen configuration

One-factor-at-a-time comparison against the baseline (gensim defaults with Skip-gram), scored with the logistic regression probe on the validation set. Baseline: macro F1 0.4047 in 518 s. Each run used one seed, so differences of about ±0.002 are treated as noise. `sample` and `window` interact.

1. **`sg=1`**: Skip-gram score beats CBOW (0.4047 vs 0.3938), which outweighs CBOW's speed (229 s).
2. **`vector_size=300`** (default 100): the only clear gain, +0.011 (0.4154).
3. **`window=3`** (default 5): faster (375 s), no tangible loss (0.4030).
4. **`min_count=5`** (default): minimal effect.
5. **`negative=2`** (default 5): faster (284 s), no gain from 10.
6. **`epochs=5`** (default): 10 epochs give +0.002 (noise) for about 1.9x the time.
7. **`sample=1e-4`** (default 1e-3): +0.001 (noise) but faster (351 s).
8. **`workers`**: 3 during development, 1 for the submission (reproducibility).
9. **`seed=80085`**.

These are applied to the final Word2Vec pipeline with a Neural Network.

| Changed | Vocab | Accuracy | Macro F1 | Weighted F1 | Seconds |
|---|---|---|---|---|---|
| default | 44,448 | 0.4628 | 0.4047 | 0.4745 | 518 |
| sg=0 (CBOW) | 44,448 | 0.4541 | 0.3938 | 0.4666 | 229 |
| vector_size=50 | 44,448 | 0.4538 | 0.3968 | 0.4651 | 552 |
| window=3 | 44,448 | 0.4617 | 0.4030 | 0.4750 | 375 |
| min_count=2 | 79,174 | 0.4625 | 0.4037 | 0.4751 | 538 |
| negative=2 | 44,448 | 0.4635 | 0.4050 | 0.4760 | 284 |
| epochs=2 | 44,448 | 0.4595 | 0.4005 | 0.4727 | 242 |
| sample=0.0001 | 44,448 | 0.4642 | 0.4059 | 0.4762 | 351 |
| vector_size=300 | 44,448 | 0.4741 | 0.4154 | 0.4867 | 829 |
| window=10 | 44,448 | 0.4644 | 0.4069 | 0.4759 | 825 |
| min_count=10 | 30,177 | 0.4622 | 0.4036 | 0.4747 | 476 |
| negative=10 | 44,448 | 0.4617 | 0.4041 | 0.4740 | 848 |
| epochs=10 | 44,448 | 0.4647 | 0.4069 | 0.4763 | 985 |
| sample=0.01 | 44,448 | 0.4628 | 0.4042 | 0.4751 | 573 |